In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(1)

producer.flush()
producer.close()

Overwriting producer.py


In [2]:
%%file consumer_anomaly.py

from kafka import KafkaConsumer
import json
from collections import defaultdict
from datetime import datetime, timedelta

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    group_id='speed-anomaly-group'
)

user_transactions = defaultdict(list)

for message in consumer:
    tx = message.value

    user_id = tx['user_id']
    timestamp = datetime.fromisoformat(tx['timestamp'])

    user_transactions[user_id].append(timestamp)

    cutoff = timestamp - timedelta(seconds=60)

    user_transactions[user_id] = [
        t for t in user_transactions[user_id]
        if t >= cutoff
    ]

    count = len(user_transactions[user_id])

    if count > 3:
        print(
            f"ALERT: user_id={user_id} | "
            f"{count} transakcje w ciągu 60 sekund"
        )

Writing consumer_anomaly.py
